<a href="https://colab.research.google.com/github/gaborh0808/1st-PyCrawlerMarathon/blob/master/Half_Kelly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import gc
import logging
import time
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_sample_weight

# 關閉 yfinance 底層訊息
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

# 1. 環境相容性設定
try:
    from google.colab import files
    HAS_COLAB = True
except ImportError:
    HAS_COLAB = False

try:
    from IPython.display import display
except ImportError:
    def display(df):
        print(df.to_string())

warnings.filterwarnings("ignore")

# 設定預測信心門檻
CONFIDENCE_THRESHOLD = 0.55

# ==============================================================================
# 0. 股票資料池定義
# ==============================================================================
stock_dict = {
    "0050.TW": "元大台灣50", "0056.TW": "元大高股息", "00878.TW": "國泰永續高股息", "00770.TW": "國泰北美科技",
    "00981A.TW": "統一台股增長主動式", "SPCX": "SPACs ETF", "SOXX": "iShares半導體ETF", "SMH": "VanEck半導體ETF",
    "AAPL": "Apple 蘋果", "GOOG": "Google / Alphabet", "META": "Meta", "MSFT": "Microsoft 微軟",
    "NVDA": "NVIDIA 輝達", "TSM": "台積電 ADR", "TSLA": "Tesla 特斯拉", "ENTG": "Entegris 英特格",
    "SMR": "NuScale Power 小型核反應爐", "BE": "Bloom Energy 燃料電池", "JNJ": "Johnson & Johnson 嬌生",
    "ASML": "ASML 艾司摩爾", "AMAT": "Applied Materials 應用材料", "LRCX": "Lam Research 柯林研發",
    "KLAC": "KLA 科磊", "AMD": "AMD 超微", "AVGO": "Broadcom 博通", "QCOM": "Qualcomm 高通",
    "INTC": "Intel 英特爾", "MU": "Micron 鎂光", "TXN": "Texas Instruments 德州儀器", "ARM": "ARM 晶心/安謀",
    "MRVL": "Marvell 邁威爾", "ADI": "Analog Devices 亞德諾", "MPWR": "Monolithic Power 芯源系統",
    "ON": "ON Semiconductor 安森美", "SWKS": "Skyworks 思佳訊", "QRVO": "Qorvo 威訊", "TER": "Teradyne 泰瑞達",
    "MKSI": "MKS Instruments", "PANW": "Palo Alto Networks", "CRWD": "CrowdStrike", "FTNT": "Fortinet",
    "NET": "Cloudflare", "ZS": "Zscaler", "OKTA": "Okta", "S": "SentinelOne", "GEN": "Gen Digital",
    "RPD": "Rapid7", "CBRS": "CyberArk", "2471.TW": "資通", "2480.TW": "敦陽科", "3029.TW": "零壹",
    "6214.TW": "精誠", "3130.TW": "一零四", "2427.TW": "三商電", "3027.TW": "盛達", "5203.TW": "訊連",
    "5471.TW": "松翰", "5410.TW": "國統", "6183.TW": "關貿", "6203.TWO": "海韻電", "6210.TWO": "慶生",
    "6593.TWO": "台灣銘板", "6689.TW": "伊雲谷", "6690.TWO": "安碁資訊", "6752.TWO": "睿嘉",
    "6763.TWO": "綠界科技", "6865.TWO": "偉康科技", "6874.TWO": "倍力", "6928.TW": "全達",
    "2382.TW": "廣達", "3231.TW": "緯創", "6669.TW": "緯穎", "2317.TW": "鴻海", "2356.TW": "英業達",
    "2324.TW": "仁寶", "2376.TW": "技嘉", "3706.TW": "神達", "2377.TW": "微 MSI", "2357.TW": "華碩",
    "4938.TW": "和碩", "3005.TW": "神基", "2353.TW": "宏碁", "2330.TW": "台積電", "2303.TW": "聯電",
    "2454.TW": "聯發科", "3034.TW": "聯詠", "3661.TW": "世芯-KY", "3443.TW": "創意", "4961.TW": "天鈺",
    "6415.TW": "矽力-KY", "6531.TW": "愛普*", "3035.TW": "智原", "6643.TWO": "M31", "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩", "6104.TWO": "創唯", "6756.TW": "威鋒電子", "2342.TW": "茂矽", "6770.TW": "力積電",
    "3707.TWO": "漢磊", "3016.TW": "嘉晶", "3711.TW": "日月光投控", "2449.TW": "京元電子", "6257.TW": "矽格",
    "3264.TWO": "欣銓", "6239.TW": "力成", "2329.TW": "華泰", "2441.TW": "超豐", "3131.TWO": "弘塑",
    "3583.TW": "辛耘", "6187.TWO": "萬潤", "2467.TW": "志聖", "8027.TWO": "钛昇", "5434.TW": "崇越",
    "3010.TW": "華立", "1560.TW": "中砂", "3680.TWO": "家登", "5234.TW": "達興材料", "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體", "6515.TW": "穎崴", "6683.TWO": "雍智科技", "6510.TWO": "精測",
    "6223.TWO": "旺矽", "2404.TW": "漢唐", "1773.TW": "勝一", "6196.TW": "帆宣", "6139.TW": "亞翔",
    "6613.TWO": "朋億*", "4755.TW": "三福化", "4768.TWO": "晶呈科技", "3563.TW": "牧德", "3167.TW": "大量",
    "6438.TW": "迅得", "1595.TWO": "川寶", "6147.TWO": "頎邦", "8150.TW": "南茂", "6552.TW": "易華電",
    "5536.TWO": "聖暉*", "3644.TWO": "凌嘉科", "7769.TW": "鴻勁", "2344.TW": "華邦電", "2408.TW": "南亞科",
    "2337.TW": "旺宏", "3006.TW": "晶豪科", "3260.TWO": "威剛", "2451.TW": "創見", "4967.TW": "十銓",
    "8271.TW": "宇瞻", "5289.TWO": "宜晶", "8299.TWO": "群聯", "5351.TWO": "鈺創", "2308.TW": "台達電",
    "2301.TW": "光寶科", "6282.TW": "康舒", "6412.TW": "群電", "3665.TW": "貿聯-KY", "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻", "3653.TW": "健策", "2421.TW": "建準", "8996.TW": "高力", "3483.TWO": "力致",
    "6230.TW": "尼得科超眾", "3013.TW": "晟銘電", "6805.TW": "富世達", "8210.TW": "勤誠", "6117.TW": "迎廣",
    "6235.TW": "華孚", "2354.TW": "鴻準", "3376.TW": "新日興", "3548.TWO": "兆利", "5243.TW": "乙盛-KY",
    "6715.TW": "嘉基", "3533.TW": "嘉澤", "3217.TWO": "優群", "3023.TW": "信邦", "2392.TW": "正崴",
    "3689.TWO": "湧德", "3357.TWO": "臺慶科", "6862.TW": "三集瑞-KY", "6821.TWO": "聯寶", "3207.TWO": "耀勝",
    "6197.TW": "佳必琪", "8103.TW": "瀚荃", "3526.TWO": "凡甲", "3605.TW": "宏致", "2059.TW": "川湖",
    "6584.TWO": "南俊國際", "2327.TW": "國巨", "2492.TW": "華新科", "2375.TW": "凱美", "2478.TW": "大毅",
    "3026.TW": "禾伸堂", "3090.TW": "日電貿", "6173.TWO": "信昌電", "6155.TW": "鈞寶", "6175.TWO": "立敦",
    "5328.TWO": "華容", "3236.TWO": "千如", "8043.TWO": "蜜望實", "3037.TW": "欣興", "8046.TW": "南電",
    "3189.TW": "景碩", "4958.TW": "臻鼎-KY", "2368.TW": "金像電", "3044.TW": "健鼎", "2313.TW": "華通",
    "8155.TWO": "博智", "2383.TW": "台光電", "6274.TWO": "台燿", "6213.TW": "聯茂", "1717.TW": "長興",
    "1815.TWO": "富喬", "1802.TW": "台玻", "5340.TWO": "建榮", "5475.TWO": "德宏", "3305.TW": "昇貿",
    "3631.TWO": "晟楠", "8358.TWO": "金居", "8021.TW": "尖點", "6672.TW": "騰輝電子-KY", "2345.TW": "智邦",
    "5388.TW": "中磊", "3558.TWO": "神準", "3704.TW": "合勤控", "4906.TW": "正文", "4979.TWO": "華星光",
    "6442.TW": "光聖", "4908.TWO": "前鼎", "3163.TWO": "波若威", "3450.TW": "聯鈞", "6426.TW": "統新",
    "4977.TW": "眾達-KY", "6530.TWO": "創威", "3363.TWO": "上詮", "3234.TWO": "光環", "4903.TWO": "聯光通",
    "3081.TWO": "聯亞", "4991.TWO": "環宇-KY", "4971.TWO": "IET-KY", "6588.TWO": "東典光電", "3491.TWO": "昇達科",
    "2314.TW": "台揚", "6285.TW": "啟碁", "3105.TWO": "穩懋", "2455.TW": "全新", "3138.TW": "耀登",
    "2419.TW": "仲琦", "2395.TW": "研華", "6166.TW": "凌華", "8050.TWO": "廣積", "3556.TWO": "禾瑞亞",
    "2414.TW": "精技", "6414.TW": "樺漢", "3022.TW": "威強電", "2397.TW": "友通", "5314.TWO": "世紀",
    "6781.TW": "AES-KY", "3211.TWO": "順達", "6121.TWO": "新普", "3323.TWO": "加百裕", "3625.TWO": "西勝",
    "8038.TWO": "長園科", "4931.TWO": "新盛力", "1519.TW": "華城", "1513.TW": "中興電", "1514.TW": "亞力",
    "1503.TW": "士電", "1609.TW": "大亞", "1605.TW": "華新", "1608.TW": "華榮", "6869.TW": "雲豹能源",
    "2049.TW": "上銀", "4576.TW": "大銀微系統", "4585.TW": "達明", "2359.TW": "所羅門", "6188.TWO": "廣明",
    "8374.TW": "羅昇", "5443.TWO": "均豪", "6640.TWO": "均華", "2464.TW": "盟立", "6215.TW": "和椿",
    "4562.TW": "穎漢", "1590.TW": "亞德客-KY", "1504.TW": "東元", "3481.TW": "群創", "2409.TW": "友達",
    "3008.TW": "大立光", "4915.TW": "先進光", "5288.TW": "匯鑽科", "2393.TW": "億光", "2201.TW": "裕隆",
    "2204.TW": "中華", "2206.TW": "三陽工業", "1536.TW": "和大", "2231.TW": "聯嘉", "3552.TWO": "同致",
    "6279.TWO": "胡連", "2603.TW": "長榮", "2609.TW": "陽明", "2615.TW": "萬海", "2605.TW": "新興",
    "2606.TW": "裕民", "2612.TW": "中航", "2617.TW": "台航", "2637.TW": "慧洋-KY", "2641.TWO": "正德",
    "5608.TW": "四維航", "2610.TW": "華航", "2618.TW": "長榮航", "2630.TW": "亞航", "5603.TWO": "陸海",
    "2607.TW": "勞運", "2608.TW": "嘉里大榮", "2611.TW": "志信", "2613.TW": "中櫃", "2636.TW": "台驊投控",
    "2642.TW": "宅配通", "2633.TW": "台灣高鐵", "5607.TW": "遠雄港", "5609.TWO": "中菲行", "8367.TW": "建新國際",
    "2892.TW": "第一金", "5880.TW": "合庫金", "1210.TW": "大成", "1215.TW": "卜蜂", "1216.TW": "統一",
    "2912.TW": "統一超", "5903.TWO": "全家", "1303.TW": "南亞", "2465.TW": "麗臺", "8163.TW": "達方",
    "3042.TW": "晶技", "8182.TWO": "加高", "3229.TW": "泰藝", "3308.TW": "聯傑", "6284.TW": "佳邦",
    "2484.TW": "希華", "8088.TWO": "華信科"
}

# ==============================================================================
# 1. 輔助數據爬蟲與正規化
# ==============================================================================
def fetch_twse_foreign_bulk(date_str):
    url = f"https://www.twse.com.tw/rwd/zh/fund/T86?date={date_str}&selectType=ALLBUT0999&response=json"
    try:
        res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5)
        if res.status_code == 200:
            js = res.json()
            if js.get("stat") == "OK" and "data" in js:
                return pd.DataFrame(js["data"], columns=js["fields"])
    except Exception:
        pass
    return None

def fetch_tpex_foreign_bulk(date_str_slash):
    url = f"https://www.tpex.org.tw/www/zh-tw/insti/qfiiStat?type=Daily&date={date_str_slash}&searchType=buy&id=&response=json"
    try:
        res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5)
        if res.status_code == 200:
            js = res.json()
            if "tables" in js and len(js["tables"]) > 0:
                t = js["tables"][0]
                return pd.DataFrame(t["data"], columns=t["fields"])
    except Exception:
        pass
    return None

def sanitize_df(df):
    if df is None or df.empty:
        return None
    d = df.copy()
    if d.index.tz is not None:
        d.index = d.index.tz_localize(None)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(-1)

    col_map = {c: str(c).strip().title().replace("Adj Close", "Close") for c in d.columns}
    d = d.rename(columns=col_map)
    d = d.loc[:, ~d.columns.duplicated()]  # 安全移除重複的 Close 欄位

    needed = ["Open", "Close", "High", "Low", "Volume"]
    return d[needed].dropna(subset=["Close"]) if all(k in d.columns for k in needed) else None

def compute_market_features(market_df):
    m = sanitize_df(market_df)
    if m is None or len(m) < 20:
        return pd.DataFrame()
    m_returns = m["Close"].pct_change()
    return pd.DataFrame({
        "Market_Vol_20": m_returns.rolling(20).std() * np.sqrt(252),
        "Market_Ret_20": m["Close"].pct_change(20),
        "Market_MA_Dist": (m["Close"] - m["Close"].rolling(20).mean()) / (m["Close"].rolling(20).mean() + 1e-6),
        "Market_Close": m["Close"],
    }, index=m.index)

def get_market_data():
    try:
        tw_feats = compute_market_features(yf.download("^TWII", period="3y", progress=False, auto_adjust=True))
    except Exception:
        tw_feats = compute_market_features(yf.download("0050.TW", period="3y", progress=False, auto_adjust=True))
    try:
        us_feats = compute_market_features(yf.download("^GSPC", period="3y", progress=False, auto_adjust=True))
    except Exception:
        us_feats = compute_market_features(yf.download("SPY", period="3y", progress=False, auto_adjust=True))
    return tw_feats, us_feats

def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))

def download_stock_with_auto_suffix(ticker, period="2y"):
    pure_t = ticker.split(".")[0]
    candidates = [f"{pure_t}.TWO", f"{pure_t}.TW", ticker] if pure_t.isdigit() and len(pure_t) >= 4 else [ticker]
    candidates = list(dict.fromkeys(candidates))

    for cand in candidates:
        try:
            df = yf.download(cand, period=period, progress=False, auto_adjust=True)
            cleaned_df = sanitize_df(df)
            if cleaned_df is not None and len(cleaned_df) > 50:
                return cleaned_df, cand
        except Exception:
            continue
    return None, ticker

# ==============================================================================
# 2. 特徵工程與 Triple Barrier 標籤演算法
# ==============================================================================
print("📌 步驟零：下載雙市場大盤行情數據並準備個股資料庫...")
tw_market_feats, us_market_feats = get_market_data()

print("正在預先抓取台股交易日外資籌碼資料...")
dummy_df, _ = download_stock_with_auto_suffix("2330.TW", period="2y")
recent_dates = dummy_df.index[-30:] if dummy_df is not None else []
twse_foreign_cache, tpex_foreign_cache = {}, {}

for r_date in recent_dates:
    date_key = r_date.strftime("%Y-%m-%d")
    d_str, d_slash = r_date.strftime("%Y%m%d"), r_date.strftime("%Y/%m/%d")
    df_twse, df_tpex = fetch_twse_foreign_bulk(d_str), fetch_tpex_foreign_bulk(d_slash)
    if df_twse is not None:
        twse_foreign_cache[date_key] = {
            str(row.iloc[0]).strip(): float(str(row.iloc[4]).replace(",", ""))
            for _, row in df_twse.iterrows() if str(row.iloc[4]).replace(",", "").replace("-", "").isdigit()
        }
    if df_tpex is not None:
        tpex_foreign_cache[date_key] = {
            str(row.iloc[0]).strip(): float(str(row.iloc[4]).replace(",", ""))
            for _, row in df_tpex.iterrows() if str(row.iloc[4]).replace(",", "").replace("-", "").isdigit()
        }
    time.sleep(0.1)

def compute_features(df, market_feats, is_tw_stock=False, ticker=None, pt_sl=[1.5, 1.0], num_days=5):
    d = df.copy()
    if d is None or len(d) < 200:
        return None, []

    x = np.arange(5)
    x_dev = x - x.mean()
    x_var = (x_dev**2).sum()
    calc_slope_5 = lambda y: (x_dev * (y - y.mean())).sum() / x_var
    d["Close_Slope"] = d["Close"].rolling(5).apply(calc_slope_5, raw=True) / (d["Close"] + 1e-6)

    tr = pd.concat([d["High"] - d["Low"], (d["High"] - d["Close"].shift(1)).abs(), (d["Low"] - d["Close"].shift(1)).abs()], axis=1).max(axis=1)
    d["NATR"] = tr.rolling(14).mean() / (d["Close"] + 1e-6)

    ma20, ma60 = d["Close"].rolling(20).mean(), d["Close"].rolling(60).mean()
    d["BB_Bandwidth"] = (4 * d["Close"].rolling(20).std()) / (ma20 + 1e-6)
    d["BIAS_5"] = (d["Close"] - d["Close"].rolling(5).mean()) / (d["Close"].rolling(5).mean() + 1e-6)

    ema12, ema26 = d["Close"].ewm(span=12).mean(), d["Close"].ewm(span=26).mean()
    d["MACD_Hist"] = (ema12 - ema26 - (ema12 - ema26).ewm(span=9).mean()) / (d["Close"] + 1e-6)
    d["MACD_Hist_Slope"] = d["MACD_Hist"].diff(3)

    d["Volume_Explosion"] = np.clip(d["Volume"] / (d["Volume"].rolling(5).mean() + 1e-6), 0, 10)
    d["Turnover_Rate"] = np.clip(d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6), 0, 10)
    d["Body_Ratio"] = (d["Close"] - d["Open"]).abs() / (d["High"] - d["Low"] + 1e-6)
    d["Upper_Shadow_Ratio"] = (d["High"] - d[["Open", "Close"]].max(axis=1)) / (d["High"] - d["Low"] + 1e-6)

    d["RSI_14"] = compute_rsi(d["Close"], 14) / 100.0
    d["RSI_Slope"] = d["RSI_14"].diff(3)

    # 籌碼面特徵賦值（嚴格精確對齊）
    f_net_series = pd.Series(0.0, index=d.index)
    if is_tw_stock and ticker:
        pure_ticker = ticker.split(".")[0]
        for rd in d.index:
            date_key = rd.strftime("%Y-%m-%d")
            if ticker.endswith(".TW") and date_key in twse_foreign_cache:
                f_net_series.loc[rd] = twse_foreign_cache[date_key].get(pure_ticker, 0.0)
            elif ticker.endswith(".TWO") and date_key in tpex_foreign_cache:
                f_net_series.loc[rd] = tpex_foreign_cache[date_key].get(pure_ticker, 0.0)

    d["Foreign_Net_Vol_Ratio"] = f_net_series / (d["Volume"] + 1e-6)
    d["Foreign_Net_MA5"] = d["Foreign_Net_Vol_Ratio"].rolling(5).mean().fillna(0)

    # 大盤特徵對齊
    m_re = market_feats.reindex(d.index).ffill().bfill()
    d["Alpha_5d"] = d["Close"].pct_change(5) - m_re["Market_Close"].pct_change(5)
    d["Market_Vol_20"], d["Market_Ret_20"], d["Market_MA_Dist"] = m_re["Market_Vol_20"], m_re["Market_Ret_20"], m_re["Market_MA_Dist"]

    d["Filter_Pass"] = (((ma20 - ma20.rolling(5).mean()).abs() / ma20 <= 0.04) &
                        ((d["Close"] * d["Volume"]).rolling(5).mean() >= (30_000_000 if is_tw_stock else 2_000_000)) &
                        (d["Close"] > ma60))

    returns = d["Close"].pct_change()
    volatility = returns.ewm(span=20).std().fillna(0)
    d["EWMA_Vol"] = volatility

    # Triple Barrier 標籤生成
    labels = []
    close = d["Close"]
    for i in range(len(close) - num_days):
        price_t0 = close.iloc[i]
        vol = volatility.iloc[i]
        if vol == 0 or pd.isna(vol):
            labels.append(np.nan)
            continue

        upper_barrier = price_t0 * (1 + pt_sl[0] * vol)
        lower_barrier = price_t0 * (1 - pt_sl[1] * vol)
        window = close.iloc[i + 1 : i + 1 + num_days]

        touch_upper = window[window >= upper_barrier].index
        touch_lower = window[window <= lower_barrier].index

        first_upper = touch_upper[0] if len(touch_upper) > 0 else pd.Timestamp.max
        first_lower = touch_lower[0] if len(touch_lower) > 0 else pd.Timestamp.max

        if first_upper == pd.Timestamp.max and first_lower == pd.Timestamp.max:
            labels.append(1)
        elif first_upper < first_lower:
            labels.append(2)
        else:
            labels.append(0)

    labels.extend([np.nan] * num_days)
    d["Target"] = labels

    f_cols = [
        "Close_Slope", "NATR", "BB_Bandwidth", "BIAS_5", "Volume_Explosion", "Turnover_Rate",
        "Body_Ratio", "Upper_Shadow_Ratio", "RSI_14", "RSI_Slope", "MACD_Hist", "MACD_Hist_Slope",
        "Alpha_5d", "Market_Vol_20", "Market_Ret_20", "Market_MA_Dist", "Foreign_Net_Vol_Ratio", "Foreign_Net_MA5"
    ]
    return d, f_cols

all_dfs, tickers = [], list(stock_dict.keys())
print("正在逐一載入並自動修正股票代號後綴...")

for t in tickers:
    try:
        # 將 period 改為 2y 確保超過 200 筆交易日
        df, corrected_ticker = download_stock_with_auto_suffix(t, period="2y")
        if df is not None:
            is_tw = corrected_ticker.endswith(".TW") or corrected_ticker.endswith(".TWO")
            df_feat, f_cols = compute_features(df, tw_market_feats if is_tw else us_market_feats, is_tw, corrected_ticker)
            if df_feat is not None:
                df_feat["Ticker"] = corrected_ticker
                df_feat["Stock_Name"] = stock_dict.get(t, stock_dict.get(corrected_ticker, corrected_ticker))
                all_dfs.append(df_feat)
    except Exception as e:
        continue

# ----------------- 關鍵新增：檢查是否有成功載入資料 -----------------
if len(all_dfs) == 0:
    raise ValueError(
        "❌ 沒有任何股票成功載入資料！\n"
        "可能原因：\n"
        "1. yfinance 被暫時封鎖 IP，請稍後再試。\n"
        "2. period='1y' 導致資料少於 200 列被過濾，請改用 period='2y'。\n"
        "3. 網路連線無法連至 Yahoo Finance。"
    )
# ------------------------------------------------------------------

panel_df = pd.concat(all_dfs).sort_index()
unique_dates = panel_df.index.unique().sort_values()
cutoff_date = unique_dates[-6]
panel_clean = panel_df.dropna(subset=f_cols)
hist_filtered = panel_clean[(panel_clean.index <= cutoff_date) & panel_clean["Target"].notnull() & panel_clean["Filter_Pass"]].copy()

# ==============================================================================
# 3. Purged Walk-Forward 訓練與歷史勝率計算
# ==============================================================================
print("📌 步驟一：執行模型滾動驗證與歷史勝率計算 (Purged Cross Validation)...")
tscv = TimeSeriesSplit(n_splits=5)
hist_filtered["OOF_Prob"] = np.nan
unique_hist_dates = hist_filtered.index.unique().sort_values()

lgb_params = {
    'objective': 'multiclass',
    'num_class': 3,
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 15,
    'max_depth': 4,
    'verbose': -1,
    'random_state': 42
}

for tr_idx, va_idx in tscv.split(unique_hist_dates):
    tr_dates, va_dates = unique_hist_dates[tr_idx], unique_hist_dates[va_idx]
    tr_dates_purged = tr_dates[:-5] if len(tr_dates) > 5 else tr_dates

    tr_mask = hist_filtered.index.isin(tr_dates_purged)
    va_mask = hist_filtered.index.isin(va_dates)

    X_tr, y_tr = hist_filtered.loc[tr_mask, f_cols], hist_filtered.loc[tr_mask, "Target"].astype(int)
    X_va = hist_filtered.loc[va_mask, f_cols]

    if len(X_tr) > 0 and len(np.unique(y_tr)) == 3:
        sample_weights = compute_sample_weight('balanced', y_tr)
        train_data = lgb.Dataset(X_tr, label=y_tr, weight=sample_weights)
        model = lgb.train(lgb_params, train_data, num_boost_round=100)

        preds = model.predict(X_va)
        if preds.ndim == 2 and preds.shape[1] == 3:
            hist_filtered.loc[va_mask, "OOF_Prob"] = preds[:, 2]

    gc.collect()

# 計算個股歷史勝率
hist_high_conf = hist_filtered[hist_filtered["OOF_Prob"] >= CONFIDENCE_THRESHOLD].copy()
stock_stats = pd.DataFrame(columns=["Ticker", "樣本數", "個股歷史勝率"])

if not hist_high_conf.empty:
    hist_high_conf["Is_Profit"] = (hist_high_conf["Target"] == 2).astype(int)
    stock_stats = hist_high_conf.groupby("Ticker", as_index=False).agg(
        樣本數=("Is_Profit", "count"),
        個股歷史勝率=("Is_Profit", "mean")
    )
    stock_stats["個股歷史勝率"] = stock_stats["個股歷史勝率"].map(lambda x: f"{x*100:.2f}%")

# ==============================================================================
# 4. 最終模型訓練、EV 計算與 Half-Kelly 部位配置
# ==============================================================================
print("📌 步驟二：訓練全量模型並計算期望值 (EV) 與 Half-Kelly 資本配置...")
y_full = hist_filtered["Target"].astype(int)
sample_weights_full = compute_sample_weight('balanced', y_full)
full_train_data = lgb.Dataset(hist_filtered[f_cols], label=y_full, weight=sample_weights_full)
full_model = lgb.train(lgb_params, full_train_data, num_boost_round=120)

recent_5_dates = unique_dates[-5:]
recent_df = panel_clean[panel_clean.index.isin(recent_5_dates) & panel_clean["Filter_Pass"]].copy()

if not recent_df.empty:
    pred_probs = full_model.predict(recent_df[f_cols])
    p_loss = pred_probs[:, 0]
    p_neutral = pred_probs[:, 1]
    p_profit = pred_probs[:, 2]

    recent_df["Prob_Profit"] = p_profit
    recent_df["Prob_Loss"] = p_loss

    pt_sl = [1.5, 1.0]
    cost = 0.003
    max_position_cap = 0.20
    vol = recent_df["EWMA_Vol"].values

    ev_long = vol * (p_profit * pt_sl[0] - p_loss * pt_sl[1]) - cost

    b_numerator = vol * pt_sl[0] - cost
    b_denominator = vol * pt_sl[1] + cost + 1e-9  # 加上極小值防止除以零

    kelly_weights = []
    for i in range(len(recent_df)):
        if b_denominator[i] <= 0 or b_numerator[i] <= 0:
            kelly_weights.append(0.0)
            continue

        b = b_numerator[i] / b_denominator[i]
        p_p = p_profit[i]
        p_l = p_loss[i]

        if b <= 0:
            kelly_weights.append(0.0)
            continue

        f_star = (b * p_p - p_l) / b
        half_kelly = max(0.0, f_star * 0.5)
        kelly_weights.append(min(max_position_cap, half_kelly))

    recent_df["EV_Long"] = ev_long
    recent_df["Kelly_Weight"] = kelly_weights

    top_targets = recent_df[recent_df["Prob_Profit"] >= CONFIDENCE_THRESHOLD].copy()

    if not top_targets.empty:
        top_targets["預測日期"] = top_targets.index.strftime("%Y-%m-%d")

        # 歸一化每日建議部位
        def safe_normalize(group):
            s = group.sum()
            return (group / s * 100) if s > 0 else pd.Series(0.0, index=group.index)

        top_targets["建議部位比率"] = top_targets.groupby("預測日期")["Kelly_Weight"].transform(safe_normalize)

        # 一對一合併個股勝率（包含安全處理機制）
        result_df = top_targets[
            ["預測日期", "Ticker", "Stock_Name", "Prob_Profit", "Foreign_Net_Vol_Ratio", "EV_Long", "Kelly_Weight", "建議部位比率"]
        ].merge(stock_stats[["Ticker", "個股歷史勝率"]], on="Ticker", how="left")

        result_df["個股歷史勝率"] = result_df["個股歷史勝率"].fillna("無歷史紀錄")

        result_df = result_df.rename(columns={
            "Ticker": "股票代號",
            "Stock_Name": "股票名稱",
            "Prob_Profit": "模型預測漲升機率",
            "Foreign_Net_Vol_Ratio": "外資買賣超比",
            "EV_Long": "單筆期望值(EV)",
            "Kelly_Weight": "半凱利建議下注(%)"
        })

        result_df["模型預測漲升機率"] = result_df["模型預測漲升機率"].map(lambda x: f"{x*100:.2f}%")
        result_df["外資買賣超比"] = result_df["外資買賣超比"].map(lambda x: f"{x*100:.2f}%")
        result_df["單筆期望值(EV)"] = result_df["單筆期望值(EV)"].map(lambda x: f"{x*100:.2f}%")
        result_df["半凱利建議下注(%)"] = result_df["半凱利建議下注(%)"].map(lambda x: f"{x*100:.2f}%")
        result_df["建議部位比率"] = result_df["建議部位比率"].map(lambda x: f"{x:.1f}%")

        output_cols = [
            "預測日期", "股票代號", "股票名稱", "模型預測漲升機率",
            "外資買賣超比", "個股歷史勝率", "單筆期望值(EV)", "半凱利建議下注(%)", "建議部位比率"
        ]
        result_df = result_df[output_cols].sort_values(by=["預測日期", "模型預測漲升機率"], ascending=[False, False])

        print(f"\n📊 [近五日預測與資產配置建議] (信心門檻 >= {CONFIDENCE_THRESHOLD*100:.0f}%)\n")
        display(result_df)

        fname = "recent_5days_confident_with_ev_kelly.xlsx"
        result_df.to_excel(fname, index=False)
        if HAS_COLAB:
            files.download(fname)
    else:
        print("⚠️ 近五日無符合信心門檻條件的標的。")
else:
    print("⚠️ 近五日無符合條件標的。")


📌 步驟零：下載雙市場大盤行情數據並準備個股資料庫...
正在預先抓取台股交易日外資籌碼資料...
正在逐一載入並自動修正股票代號後綴...


ValueError: ❌ 沒有任何股票成功載入資料！
可能原因：
1. yfinance 被暫時封鎖 IP，請稍後再試。
2. period='1y' 導致資料少於 200 列被過濾，請改用 period='2y'。
3. 網路連線無法連至 Yahoo Finance。